In [ ]:
# гиперпараметры
IMG_SIZE   = 96        # STL10 native resolution
BATCH_SIZE = 64
NUM_WORKERS = 2

# STL10 normalization stats (ImageNet-style приближение достаточно для базовой CNN;
# для ResNet используем точные ImageNet stats)
MEAN_STL = (0.4467, 0.4398, 0.4066)
STD_STL  = (0.2603, 0.2566, 0.2713)

MEAN_IN  = (0.485, 0.456, 0.406)
STD_IN   = (0.229, 0.224, 0.225)

# transforms
transform_base = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN_STL, STD_STL),
])

transform_aug = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(96, padding=12),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(MEAN_STL, STD_STL),
])

transform_resnet = T.Compose([
    T.Resize(224),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(MEAN_IN, STD_IN),
])

transform_resnet_aug = T.Compose([
    T.Resize(224),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(224, padding=28),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.Normalize(MEAN_IN, STD_IN),
])

In [ ]:
# загрузка STL10
# train split: 5 000 изображений, 10 классов
# test split : 8 000 изображений
# Из train вырезаем 20% как val (seed=42)

def make_stl10_loaders(train_transform, val_transform, batch_size=BATCH_SIZE):
    full_train = datasets.STL10(root=DATA_ROOT, split='train',
                                download=True, transform=train_transform)
    test_ds    = datasets.STL10(root=DATA_ROOT, split='test',
                                download=True, transform=val_transform)

    n_train = int(0.8 * len(full_train))
    n_val   = len(full_train) - n_train
    gen = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = random_split(full_train, [n_train, n_val], generator=gen)

    # val использует тот же transform, что и test (без аугментаций)
    # Применим val_transform напрямую через обёртку
    val_ds.dataset = copy.deepcopy(full_train)
    val_ds.dataset.transform = val_transform

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    return train_loader, val_loader, test_loader, n_train, n_val

# Создаём базовые загрузчики для sanity-check
train_loader_base, val_loader_base, test_loader_base, n_train, n_val = \
    make_stl10_loaders(transform_base, transform_base)

print(f'Train samples : {n_train}')
print(f'Val samples   : {n_val}')
print(f'Test samples  : {len(test_loader_base.dataset)}')

STL10_CLASSES = ['airplane', 'bird', 'car', 'cat', 'deer',
                 'dog', 'horse', 'monkey', 'ship', 'truck']
NUM_CLASSES = 10

In [ ]:
# sanity-check
xb, yb = next(iter(train_loader_base))
print(f'x.shape = {xb.shape}')   # (64, 3, 96, 96)
print(f'y.shape = {yb.shape}')   # (64,)
print(f'y values: {yb[:8].tolist()}')
print(f'x min/max: {xb.min():.3f} / {xb.max():.3f}')

In [ ]:
# визуализация примеров датасета
def denorm(t, mean, std):
    mean = torch.tensor(mean).view(3,1,1)
    std  = torch.tensor(std).view(3,1,1)
    return (t * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = denorm(xb[i], MEAN_STL, STD_STL).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(STL10_CLASSES[yb[i].item()], fontsize=9)
    ax.axis('off')
plt.suptitle('STL10 – sample images', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/stl10_samples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# визуализация аугментаций
raw_ds = datasets.STL10(root=DATA_ROOT, split='train',
                        download=False, transform=None)
aug_transform_vis = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(96, padding=12),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.1),
])
to_tensor = T.ToTensor()

fig, axes = plt.subplots(4, 5, figsize=(13, 11))
for row, idx in enumerate([0, 1, 2, 3]):
    orig, label = raw_ds[idx]
    axes[row, 0].imshow(orig)
    axes[row, 0].set_title(f'Original\n{STL10_CLASSES[label]}', fontsize=8)
    axes[row, 0].axis('off')
    for col in range(1, 5):
        aug = aug_transform_vis(orig)
        axes[row, col].imshow(aug)
        axes[row, col].set_title(f'Aug #{col}', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('STL10 – augmentation preview', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/augmentations_preview.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# SimpleCNN (используется для C1 и C2)
class SimpleCNN(nn.Module):
    """Простая сверточная сеть для STL10 (96x96)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 96→48
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.1),

            # Block 2: 48→24
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.15),

            # Block 3: 24→12
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),

            # Block 4: 12→6
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 256), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ResNet18 builders
def build_resnet18_head_only(num_classes=10):
    """C3: замораживаем backbone, обучаем только fc."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_resnet18_finetune(num_classes=10):
    """C4: размораживаем layer4 + fc, остальное заморожено."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# пробный print
cnn_test = SimpleCNN(NUM_CLASSES)
total_params = sum(p.numel() for p in cnn_test.parameters())
print(f'SimpleCNN total params: {total_params:,}')

In [ ]:
# train / eval functions
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out  = model(xb)
        loss = criterion(out, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total


def run_training(model, train_loader, val_loader, optimizer, scheduler=None,
                 max_epochs=30, patience=7, device=DEVICE, exp_id='EXP'):
    """Цикл обучения с early stopping по val_accuracy."""
    model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state  = None
    wait        = 0

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device)
        vl_loss, vl_acc = evaluate(model, val_loader, device)

        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f'[{exp_id}] Epoch {epoch:3d}/{max_epochs}  '
                  f'tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  '
                  f'vl_loss={vl_loss:.4f}  vl_acc={vl_acc:.4f}')

        if wait >= patience:
            print(f'[{exp_id}] Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    print(f'[{exp_id}] Best val_acc = {best_val_acc:.4f}')
    return model, history, best_val_acc, epoch

In [ ]:
MAX_EPOCHS = 40
PATIENCE   = 8
LR_CNN     = 3e-4
LR_HEAD    = 1e-3
LR_FINE    = 1e-4

results = {}   # dict[exp_id] = {history, best_val_acc, test_acc, epochs_trained}

In [ ]:
# C1: SimpleCNN без аугментаций
print('=' * 60)
print('C1  simple-cnn-base  (no augmentation)')
print('=' * 60)

torch.manual_seed(SEED)
train_l, val_l, test_l, _, _ = make_stl10_loaders(transform_base, transform_base)

c1_model = SimpleCNN(NUM_CLASSES)
c1_opt   = optim.Adam(c1_model.parameters(), lr=LR_CNN, weight_decay=1e-4)
c1_sched = optim.lr_scheduler.CosineAnnealingLR(c1_opt, T_max=MAX_EPOCHS)

c1_model, c1_hist, c1_val_acc, c1_epochs = run_training(
    c1_model, train_l, val_l, c1_opt, c1_sched,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, exp_id='C1'
)
_, c1_test_acc = evaluate(c1_model, test_l, DEVICE)
results['C1'] = dict(history=c1_hist, best_val_acc=c1_val_acc,
                     test_acc=c1_test_acc, epochs=c1_epochs)
print(f'C1  test_acc = {c1_test_acc:.4f}')

In [ ]:
# C2: SimpleCNN с аугментациями
print('=' * 60)
print('C2  simple-cnn-aug  (with augmentation)')
print('=' * 60)

torch.manual_seed(SEED)
train_l, val_l, test_l, _, _ = make_stl10_loaders(transform_aug, transform_base)

c2_model = SimpleCNN(NUM_CLASSES)
c2_opt   = optim.Adam(c2_model.parameters(), lr=LR_CNN, weight_decay=1e-4)
c2_sched = optim.lr_scheduler.CosineAnnealingLR(c2_opt, T_max=MAX_EPOCHS)

c2_model, c2_hist, c2_val_acc, c2_epochs = run_training(
    c2_model, train_l, val_l, c2_opt, c2_sched,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, exp_id='C2'
)
_, c2_test_acc = evaluate(c2_model, test_l, DEVICE)
results['C2'] = dict(history=c2_hist, best_val_acc=c2_val_acc,
                     test_acc=c2_test_acc, epochs=c2_epochs)
print(f'C2  test_acc = {c2_test_acc:.4f}')

In [ ]:
# C3: ResNet18 head-only
print('=' * 60)
print('C3  resnet18-head-only')
print('=' * 60)

torch.manual_seed(SEED)
train_l, val_l, test_l, _, _ = make_stl10_loaders(transform_resnet, transform_resnet)

c3_model = build_resnet18_head_only(NUM_CLASSES)
c3_opt   = optim.Adam(filter(lambda p: p.requires_grad, c3_model.parameters()),
                      lr=LR_HEAD, weight_decay=1e-4)
c3_sched = optim.lr_scheduler.StepLR(c3_opt, step_size=10, gamma=0.5)

c3_model, c3_hist, c3_val_acc, c3_epochs = run_training(
    c3_model, train_l, val_l, c3_opt, c3_sched,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, exp_id='C3'
)
_, c3_test_acc = evaluate(c3_model, test_l, DEVICE)
results['C3'] = dict(history=c3_hist, best_val_acc=c3_val_acc,
                     test_acc=c3_test_acc, epochs=c3_epochs)
print(f'C3  test_acc = {c3_test_acc:.4f}')

In [ ]:
# C4: ResNet18 fine-tune (layer4 + fc)
print('=' * 60)
print('C4  resnet18-finetune  (layer4 + fc)')
print('=' * 60)

torch.manual_seed(SEED)
train_l, val_l, test_l, _, _ = make_stl10_loaders(transform_resnet_aug, transform_resnet)

c4_model = build_resnet18_finetune(NUM_CLASSES)
c4_opt   = optim.Adam(filter(lambda p: p.requires_grad, c4_model.parameters()),
                      lr=LR_FINE, weight_decay=1e-4)
c4_sched = optim.lr_scheduler.CosineAnnealingLR(c4_opt, T_max=MAX_EPOCHS)

c4_model, c4_hist, c4_val_acc, c4_epochs = run_training(
    c4_model, train_l, val_l, c4_opt, c4_sched,
    max_epochs=MAX_EPOCHS, patience=PATIENCE, exp_id='C4'
)
_, c4_test_acc = evaluate(c4_model, test_l, DEVICE)
results['C4'] = dict(history=c4_hist, best_val_acc=c4_val_acc,
                     test_acc=c4_test_acc, epochs=c4_epochs)
print(f'C4  test_acc = {c4_test_acc:.4f}')

In [ ]:
# Выбор лучшей модели и финальный тест
best_exp = max(results, key=lambda k: results[k]['best_val_acc'])
best_info = results[best_exp]
print(f'Best experiment   : {best_exp}')
print(f'Best val_accuracy : {best_info["best_val_acc"]:.4f}')
print(f'Test accuracy     : {best_info["test_acc"]:.4f}')

# Сохраняем state_dict лучшей модели
best_model_map = {'C1': c1_model, 'C2': c2_model, 'C3': c3_model, 'C4': c4_model}
best_model     = best_model_map[best_exp]
torch.save(best_model.state_dict(), f'{ART_ROOT}/best_classifier.pt')
print(f'Saved: {ART_ROOT}/best_classifier.pt')

In [ ]:
# best_classifier_config.json
config = {
    'experiment_id': best_exp,
    'dataset': 'STL10',
    'num_classes': NUM_CLASSES,
    'seed': SEED,
    'architecture': 'resnet18-finetune' if best_exp == 'C4' else
                    'resnet18-head-only' if best_exp == 'C3' else 'SimpleCNN',
    'finetune_layers': 'layer4 + fc' if best_exp == 'C4' else
                       'fc only' if best_exp == 'C3' else 'all',
    'train_transform': 'resnet_aug' if best_exp == 'C4' else
                       'resnet_base' if best_exp == 'C3' else
                       'aug' if best_exp == 'C2' else 'base',
    'val_transform': 'resnet_base' if best_exp in ('C3', 'C4') else 'base',
    'image_size': 224 if best_exp in ('C3', 'C4') else 96,
    'batch_size': BATCH_SIZE,
    'optimizer': 'Adam',
    'lr': LR_FINE if best_exp == 'C4' else LR_HEAD if best_exp == 'C3' else LR_CNN,
    'weight_decay': 1e-4,
    'scheduler': 'CosineAnnealingLR' if best_exp in ('C1', 'C2', 'C4') else 'StepLR',
    'max_epochs': MAX_EPOCHS,
    'early_stopping_patience': PATIENCE,
    'epochs_trained': best_info['epochs'],
    'best_val_accuracy': round(best_info['best_val_acc'], 4),
    'test_accuracy': round(best_info['test_acc'], 4),
}
with open(f'{ART_ROOT}/best_classifier_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

In [ ]:
# График кривых лучшего прогона
hist = best_info['history']
epochs_range = range(1, len(hist['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs_range, hist['train_loss'], label='Train loss', color='royalblue')
ax1.plot(epochs_range, hist['val_loss'],   label='Val loss',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title(f'{best_exp} – Loss curves')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, hist['train_acc'], label='Train acc', color='royalblue')
ax2.plot(epochs_range, hist['val_acc'],   label='Val acc',   color='tomato')
ax2.axhline(best_info['test_acc'], color='green', linestyle='--', label='Test acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'{best_exp} – Accuracy curves')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle(f'Best run: {best_exp}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/classification_curves_best.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Сравнение C1–C4
exp_ids   = ['C1', 'C2', 'C3', 'C4']
val_accs  = [results[k]['best_val_acc'] for k in exp_ids]
test_accs = [results[k]['test_acc']     for k in exp_ids]

x    = np.arange(len(exp_ids))
w    = 0.35
colors_val  = ['#5B8FD1', '#5B8FD1', '#E87040', '#E87040']
colors_test = ['#2C5F9A', '#2C5F9A', '#B84A1E', '#B84A1E']

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, val_accs,  w, label='Val  acc',  color=colors_val,  alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + w/2, test_accs, w, label='Test acc',  color=colors_test, alpha=0.85, edgecolor='white')

for bar, v in zip(bars1, val_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=9)
for bar, v in zip(bars2, test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(['C1\nsimple-cnn\nbase', 'C2\nsimple-cnn\naug',
                    'C3\nresnet18\nhead-only', 'C4\nresnet18\nfinetune'])
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('C1–C4 Comparison: Val & Test Accuracy')
ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/classification_compare.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nSummary table:')
for eid in exp_ids:
    r = results[eid]
    print(f'  {eid}: val={r["best_val_acc"]:.4f}  test={r["test_acc"]:.4f}  '
          f'epochs={r["epochs"]}')

In [ ]:
# OxfordIIITPet segmentation
# Маска: 1 = pet (foreground), 2 = background, 3 = boundary
# Будем считать foreground = класс 1 (пиксели самого животного)

SEG_SIZE = 224

transform_seg_img = T.Compose([
    T.Resize((SEG_SIZE, SEG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN_IN, STD_IN),
])

transform_seg_mask = T.Compose([
    T.Resize((SEG_SIZE, SEG_SIZE), interpolation=T.InterpolationMode.NEAREST),
])

# torchvision OxfordIIITPet с target_types='segmentation'
pet_ds = datasets.OxfordIIITPet(
    root=DATA_ROOT, split='test',
    target_types='segmentation',
    download=True,
    transform=transform_seg_img,
    target_transform=transform_seg_mask,
)

print(f'OxfordIIITPet test set size: {len(pet_ds)}')

# Используем подвыборку для быстрого инференса
N_EVAL = 100
gen2   = torch.Generator().manual_seed(SEED)
eval_idx = torch.randperm(len(pet_ds), generator=gen2)[:N_EVAL].tolist()
pet_eval = Subset(pet_ds, eval_idx)
pet_loader = DataLoader(pet_eval, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# sanity-check Part B
import PIL

raw_pet = datasets.OxfordIIITPet(
    root=DATA_ROOT, split='test',
    target_types='segmentation',
    download=False,
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col in range(4):
    img, mask = raw_pet[eval_idx[col]]
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'Image #{eval_idx[col]}', fontsize=9)
    axes[0, col].axis('off')
    mask_np = np.array(mask)
    axes[1, col].imshow(mask_np, cmap='tab10', vmin=0, vmax=3)
    axes[1, col].set_title('Mask (1=fg, 2=bg, 3=border)', fontsize=8)
    axes[1, col].axis('off')
plt.suptitle('OxfordIIITPet – segmentation samples', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# DeepLabV3_ResNet50 pretrained
seg_model = deeplabv3_resnet50(
    weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
)
seg_model.eval().to(DEVICE)

# COCO class 12 = 'dog', 17 = 'cat'  → используем оба как foreground
# Маска предсказаний: argmax по 21 классу (VOC labels)
# Foreground = любой класс кроме 0 (background)
# Для OxfordIIITPet: foreground = argmax_class in {12, 17}
# В отчёте чётко объясняем: fg = «cat» OR «dog» в предсказании модели

PET_CLASSES_COCO = {12, 17}   # dog=12, cat=17 в COCO

@torch.no_grad()
def predict_mask(model, img_tensor, device):
    img_tensor = img_tensor.to(device)
    output = model(img_tensor)['out']          # (1, 21, H, W)
    pred   = output.argmax(dim=1).squeeze(0)   # (H, W)
    return pred.cpu()


def pred_to_fg(pred_mask):
    """Бинарная маска: 1 там, где модель предсказала кошку или собаку."""
    fg = torch.zeros_like(pred_mask, dtype=torch.bool)
    for cls in PET_CLASSES_COCO:
        fg |= (pred_mask == cls)
    return fg


def gt_to_fg(gt_pil_mask):
    """Ground-truth foreground: пиксели с значением 1 (pet)."""
    mask_np = np.array(gt_pil_mask.resize((SEG_SIZE, SEG_SIZE),
                                           PIL.Image.NEAREST))
    return torch.from_numpy(mask_np == 1)   # bool tensor


def compute_iou(pred_fg, gt_fg):
    inter = (pred_fg & gt_fg).sum().item()
    union = (pred_fg | gt_fg).sum().item()
    return inter / union if union > 0 else 0.0


def compute_precision_recall(pred_fg, gt_fg):
    tp = (pred_fg & gt_fg).sum().item()
    fp = (pred_fg & ~gt_fg).sum().item()
    fn = (~pred_fg & gt_fg).sum().item()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return prec, rec


print('DeepLabV3_ResNet50 loaded OK')

In [ ]:
# V1: базовая постобработка
# Просто argmax → бинарная маска fg (cat|dog) без доп. фильтрации

def postprocess_v1(pred_mask):
    return pred_to_fg(pred_mask)


v1_ious, v1_precs, v1_recs = [], [], []

raw_pet_test = datasets.OxfordIIITPet(
    root=DATA_ROOT, split='test', target_types='segmentation',
    download=False, transform=None
)

for i, (img_t, _) in enumerate(pet_loader):
    pred  = predict_mask(seg_model, img_t, DEVICE)
    fg_pred_v1 = postprocess_v1(pred)

    _, gt_pil = raw_pet_test[eval_idx[i]]
    fg_gt = gt_to_fg(gt_pil)

    v1_ious.append(compute_iou(fg_pred_v1, fg_gt))
    p, r = compute_precision_recall(fg_pred_v1, fg_gt)
    v1_precs.append(p)
    v1_recs.append(r)

v1_miou  = np.mean(v1_ious)
v1_mprec = np.mean(v1_precs)
v1_mrec  = np.mean(v1_recs)
print(f'V1 – mean IoU      : {v1_miou:.4f}')
print(f'V1 – mean Precision: {v1_mprec:.4f}')
print(f'V1 – mean Recall   : {v1_mrec:.4f}')

In [ ]:
# V2: удаление малых связных компонент
# Удаляем компоненты связности, занимающие < 2% площади изображения

MIN_AREA_FRAC = 0.02  # порог: 2% от SEG_SIZE^2
MIN_AREA_PX   = int(MIN_AREA_FRAC * SEG_SIZE * SEG_SIZE)


def postprocess_v2(pred_mask):
    fg_np = pred_to_fg(pred_mask).numpy().astype(np.uint8)
    labeled, n_comp = ndimage.label(fg_np)
    cleaned = np.zeros_like(fg_np)
    for comp_id in range(1, n_comp + 1):
        if (labeled == comp_id).sum() >= MIN_AREA_PX:
            cleaned[labeled == comp_id] = 1
    return torch.from_numpy(cleaned.astype(bool))


v2_ious, v2_precs, v2_recs = [], [], []

for i, (img_t, _) in enumerate(pet_loader):
    pred  = predict_mask(seg_model, img_t, DEVICE)
    fg_pred_v2 = postprocess_v2(pred)

    _, gt_pil = raw_pet_test[eval_idx[i]]
    fg_gt = gt_to_fg(gt_pil)

    v2_ious.append(compute_iou(fg_pred_v2, fg_gt))
    p, r = compute_precision_recall(fg_pred_v2, fg_gt)
    v2_precs.append(p)
    v2_recs.append(r)

v2_miou  = np.mean(v2_ious)
v2_mprec = np.mean(v2_precs)
v2_mrec  = np.mean(v2_recs)
print(f'V2 – mean IoU      : {v2_miou:.4f}')
print(f'V2 – mean Precision: {v2_mprec:.4f}')
print(f'V2 – mean Recall   : {v2_mrec:.4f}')

In [ ]:
# Визуализация сегментации
VIS_IDX = [0, 1, 2, 3, 4]   # первые 5 из eval subset

fig, axes = plt.subplots(len(VIS_IDX), 4, figsize=(16, 4 * len(VIS_IDX)))
fig.suptitle('OxfordIIITPet – Segmentation Results', fontsize=14)

for row, vi in enumerate(VIS_IDX):
    img_raw, gt_pil = raw_pet_test[eval_idx[vi]]

    # inference
    img_t, _ = pet_ds[vi]
    pred = predict_mask(seg_model, img_t.unsqueeze(0), DEVICE)
    fg_v1 = postprocess_v1(pred).numpy()
    fg_v2 = postprocess_v2(pred).numpy()
    fg_gt = gt_to_fg(gt_pil).numpy()

    axes[row, 0].imshow(img_raw)
    axes[row, 0].set_title('Original', fontsize=9); axes[row, 0].axis('off')

    axes[row, 1].imshow(fg_gt, cmap='Greens', vmin=0, vmax=1)
    axes[row, 1].set_title('GT foreground', fontsize=9); axes[row, 1].axis('off')

    axes[row, 2].imshow(fg_v1, cmap='Blues', vmin=0, vmax=1)
    iou_v1 = compute_iou(torch.from_numpy(fg_v1.astype(bool)),
                         torch.from_numpy(fg_gt.astype(bool)))
    axes[row, 2].set_title(f'V1 pred  IoU={iou_v1:.3f}', fontsize=9)
    axes[row, 2].axis('off')

    axes[row, 3].imshow(fg_v2, cmap='Oranges', vmin=0, vmax=1)
    iou_v2 = compute_iou(torch.from_numpy(fg_v2.astype(bool)),
                         torch.from_numpy(fg_gt.astype(bool)))
    axes[row, 3].set_title(f'V2 pred  IoU={iou_v2:.3f}', fontsize=9)
    axes[row, 3].axis('off')

plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/segmentation_examples.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Метрики V1 vs V2 (bar chart)
metrics_names = ['mean IoU', 'mean Precision', 'mean Recall']
v1_metrics = [v1_miou, v1_mprec, v1_mrec]
v2_metrics = [v2_miou, v2_mprec, v2_mrec]

x  = np.arange(len(metrics_names))
w  = 0.3
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, v1_metrics, w, label='V1 (argmax, no postproc)', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, v2_metrics, w, label='V2 (remove small comps)',  color='darkorange', alpha=0.85)
for bar, v in zip(b1, v1_metrics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=9)
for bar, v in zip(b2, v2_metrics):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.3f}',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Segmentation metrics: V1 vs V2 (OxfordIIITPet)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/segmentation_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# IoU distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(v1_ious, bins=20, alpha=0.6, label='V1 IoU per image', color='steelblue')
ax.hist(v2_ious, bins=20, alpha=0.6, label='V2 IoU per image', color='darkorange')
ax.axvline(v1_miou, color='steelblue', linestyle='--', label=f'V1 mean={v1_miou:.3f}')
ax.axvline(v2_miou, color='darkorange', linestyle='--', label=f'V2 mean={v2_miou:.3f}')
ax.set_xlabel('IoU'); ax.set_ylabel('Count')
ax.set_title('IoU distribution per image (V1 vs V2)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_ROOT}/segmentation_iou_hist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
rows = []

exp_cfgs = {
    'C1': dict(model='SimpleCNN', opt='Adam', lr=LR_CNN,
               notes='no augmentations, base transforms'),
    'C2': dict(model='SimpleCNN', opt='Adam', lr=LR_CNN,
               notes='augmentation: hflip, crop, colorjitter, grayscale'),
    'C3': dict(model='ResNet18 (head-only, pretrained ImageNet)', opt='Adam', lr=LR_HEAD,
               notes='backbone frozen; only fc trained'),
    'C4': dict(model='ResNet18 (layer4+fc finetune, pretrained ImageNet)', opt='Adam', lr=LR_FINE,
               notes='layer4+fc unfrozen; aug train transform'),
}

for eid, cfg in exp_cfgs.items():
    r = results[eid]
    rows.append({
        'experiment_id'   : eid,
        'task'            : 'classification',
        'dataset'         : 'STL10',
        'seed'            : SEED,
        'model_summary'   : cfg['model'],
        'optimizer'       : cfg['opt'],
        'lr'              : cfg['lr'],
        'epochs_trained'  : r['epochs'],
        'best_val_accuracy': round(r['best_val_acc'], 4),
        'test_accuracy'   : round(r['test_acc'], 4),
        'precision'       : '',
        'recall'          : '',
        'mean_iou'        : '',
        'notes'           : cfg['notes'],
    })

for vid, miou, mprec, mrec, note in [
    ('V1', v1_miou, v1_mprec, v1_mrec, 'argmax mask; fg = cat|dog (COCO cls 12,17)'),
    ('V2', v2_miou, v2_mprec, v2_mrec, 'remove small connected comps < 2% area'),
]:
    rows.append({
        'experiment_id'   : vid,
        'task'            : 'segmentation',
        'dataset'         : 'OxfordIIITPet',
        'seed'            : SEED,
        'model_summary'   : 'DeepLabV3_ResNet50 (COCO_VOC pretrained)',
        'optimizer'       : '',
        'lr'              : '',
        'epochs_trained'  : 0,
        'best_val_accuracy': '',
        'test_accuracy'   : '',
        'precision'       : round(mprec, 4),
        'recall'          : round(mrec, 4),
        'mean_iou'        : round(miou, 4),
        'notes'           : note,
    })

df = pd.DataFrame(rows)
df.to_csv(f'{ART_ROOT}/runs.csv', index=False)
print('Saved runs.csv')
df

In [ ]:
print('\n=== ALL ARTIFACTS SAVED ===')
for path, _, files in os.walk(ART_ROOT):
    for f in sorted(files):
        full = os.path.join(path, f)
        size = os.path.getsize(full)
        print(f'  {full}  ({size:,} bytes)')